In [4]:
# 1) 충성 사용자 리포트
# 이용자와 댓글 데이터를 결합해, 총 댓글 수가 10개 이상인 이용자만 추립니다.
# 각 이용자에 대해 다음 지표를 계산해 주세요:
# commentsCount(총 댓글 수)
# avgTextLen(댓글 평균 길이, null/missing은 길이 0으로 간주)
# lastCommentDate(가장 최근 댓글 시점)
# 결과에는 **이름(name), 이메일(email)**만 식별 정보로 포함하고, 위 활동 지표 **내림차순(많이·길게·최근 순)**으로 정렬해 상위 이용자가 누구인지 한눈에 보이도록 합니다

# 출력 형식 가이드
# 충성 사용자 리포트: 리스트(배열) 형태. 각 원소는 {name, email, commentsCount, avgTextLen, lastCommentDate}.

from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017")

db = client.sample_mflix
users = db.users

pipeline = [
    # comments 컬렉션과 조인
    {
        "$lookup": {
            "from": "comments",
            "localField": "email",
            "foreignField": "email",
            "as": "userComments"
        }
    },
    # 댓글 없는 경우도 일단 배열 유지
    {
        "$addFields": {
            "commentsCount": {"$size": "$userComments"},
            "avgTextLen": {
                "$cond": [
                    {"$gt": [{"$size": "$userComments"}, 0]},
                    {
                        "$avg": {
                            "$map": {
                                "input": "$userComments",
                                "as": "c",
                                "in": {
                                    "$strLenCP": {
                                        "$ifNull": ["$$c.text", ""]
                                    }
                                }
                            }
                        }
                    },
                    0
                ]
            },
            "lastCommentDate": {
                "$max": "$userComments.date"
            }
        }
    },
    # 댓글 10개 이상만 추리기
    {
        "$match": {"commentsCount": {"$gte": 10}}
    },
    # 필요한 필드만 투영
    {
        "$project": {
            "_id": 0,
            "name": 1,
            "email": 1,
            "commentsCount": 1,
            "avgTextLen": 1,
            "lastCommentDate": 1
        }
    },
    # 정렬 (많이,길게,최근 순)
    {
        "$sort": {
            "commentsCount": -1,
            "avgTextLen": -1,
            "lastCommentDate": -1
        }
    }
]

# 충성 사용자 리포트: 리스트(배열) 형태.
results = list(users.aggregate(pipeline))

for result in results[:10] :
    print(f"{result}")

{'name': 'Mace Tyrell', 'email': 'roger_ashton-griffiths@gameofthron.es', 'commentsCount': 331, 'avgTextLen': 152.49244712990938, 'lastCommentDate': datetime.datetime(2017, 8, 2, 1, 37, 26)}
{'name': 'Missandei', 'email': 'nathalie_emmanuel@gameofthron.es', 'commentsCount': 327, 'avgTextLen': 153.0948012232416, 'lastCommentDate': datetime.datetime(2017, 9, 11, 16, 52, 51)}
{'name': 'The High Sparrow', 'email': 'jonathan_pryce@gameofthron.es', 'commentsCount': 315, 'avgTextLen': 152.68253968253967, 'lastCommentDate': datetime.datetime(2017, 7, 10, 4, 58, 23)}
{'name': 'Sansa Stark', 'email': 'sophie_turner@gameofthron.es', 'commentsCount': 308, 'avgTextLen': 153.42532467532467, 'lastCommentDate': datetime.datetime(2017, 5, 2, 16, 29, 50)}
{'name': 'Rodrik Cassel', 'email': 'ron_donachie@gameofthron.es', 'commentsCount': 305, 'avgTextLen': 151.75737704918032, 'lastCommentDate': datetime.datetime(2017, 9, 2, 6, 1, 19)}
{'name': 'Thoros of Myr', 'email': 'paul_kaye@gameofthron.es', 'commen

In [11]:
# 2) 영화 인사이트 리포트
# 영화 컬렉션만으로 한 번의 파이프라인에서 다음을 동시에 산출해 주세요.

# 최신 영화 5편: 제목(title)과 연도(year) 목록
# 고평점 영화 개수: imdb.rating ≥ 8 작품의 총 개수(숫자 1개)
# 장르별 상위 10: 장르별 작품 수를 집계해 상위 10개를 많이 나온 순으로 정렬
# 연도별 평균 평점(최근 10년): 최신 연도를 기준으로 최근 10개 연도의 평균 평점을 산출(연도-오름차순 표기)

# 출력 형식 가이드

# 영화 인사이트 리포트: 단일 문서(오브젝트) 형태.
# 키는 latest5, highRatedCount, genresTop10, yearlyAvgRecent10로 구성.
# highRatedCount는 숫자 하나로 제출

from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017")
db = client["sample_mflix"]
movies = db["movies"]

pipeline = [
    {
        "$facet": {
            # 최신 영화 5편
            "latest5": [
                {"$sort": {"year": -1}},
                {"$limit": 5},
                {"$project": {"_id": 0, "title": 1, "year": 1}}
            ],
            # 고평점 영화 개수
            "highRatedCount": [
                {"$match": {"imdb.rating": {"$gte": 8}}},
                {"$count": "count"}
            ],
            # 장르별 상위 10
            "genresTop10": [
                {"$unwind": "$genres"},
                {"$group": {"_id": "$genres", "count": {"$sum": 1}}},
                {"$sort": {"count": -1}},
                {"$limit": 10},
                {"$project": {"_id": 0, "genre": "$_id", "count": 1}}
            ],
            # 연도별 평균 평점 (최근 10년)
            "yearlyAvgRecent10": [
                {"$group": {"_id": "$year", "avgRating": {"$avg": "$imdb.rating"}}},
                {"$sort": {"_id": -1}},
                {"$limit": 10},
                {"$sort": {"_id": 1}},
                {"$project": {"_id": 0, "year": "$_id", "avgRating": 1}}
            ]
        }
    },
    {
        # 최종 형식 맞추기
        "$project": {
            "latest5": 1,
            "highRatedCount": {"$arrayElemAt": ["$highRatedCount.count", 0]},
            "genresTop10": 1,
            "yearlyAvgRecent10": 1
        }
    }
]

# 영화 인사이트 리포트
result = list(movies.aggregate(pipeline))[0]

for key, value in result.items():
    print(f"\n=== {key} ===")
    
    if isinstance(value, list):
        for item in value:
            print(item)
    else:
        print(value)


=== latest5 ===
{'title': 'The Saboteurs', 'year': '2015è'}
{'title': 'Halo: Nightfall', 'year': '2014è'}
{'title': 'The Roosevelts: An Intimate History', 'year': '2014è'}
{'title': 'The Weight of the Nation', 'year': '2012è'}
{'title': 'Hit & Miss', 'year': '2012è'}

=== genresTop10 ===
{'count': 13789, 'genre': 'Drama'}
{'count': 7024, 'genre': 'Comedy'}
{'count': 3665, 'genre': 'Romance'}
{'count': 2678, 'genre': 'Crime'}
{'count': 2658, 'genre': 'Thriller'}
{'count': 2539, 'genre': 'Action'}
{'count': 2129, 'genre': 'Documentary'}
{'count': 2045, 'genre': 'Adventure'}
{'count': 1703, 'genre': 'Horror'}
{'count': 1404, 'genre': 'Biography'}

=== yearlyAvgRecent10 ===
{'avgRating': 7.2, 'year': '2006è'}
{'avgRating': 9.0, 'year': '2006è2007'}
{'avgRating': 8.5, 'year': '2006è2012'}
{'avgRating': 8.533333333333333, 'year': '2007è'}
{'avgRating': 8.3, 'year': '2009è'}
{'avgRating': 8.2, 'year': '2010è'}
{'avgRating': 7.75, 'year': '2011è'}
{'avgRating': 7.5, 'year': '2012è'}
{'avgRati